# Size-25 von Mises fit audit and paper-faithful interference

This notebook checks whether the von Mises fitting completed correctly for the full primary grid, excluding `0.5` sparsity:

- `two_module_rnn`, `shared`, `sp=1.0`, `init_scale in {0.001, 0.01, 0.1, 1, 2}` at size `25`
- `two_module_rnn`, `shared`, `no_comms`, `init_scale in {0.001, 0.01, 0.1, 1, 2}` at size `25`
- `two_module_rnn`, `task_routed`, `sp=1.0`, `init_scale in {0.001, 0.01, 0.1, 1, 2}` at size `25`
- `two_module_rnn`, `task_routed`, `no_comms`, `init_scale in {0.001, 0.01, 0.1, 1, 2}` at size `25`
- `single_module_rnn`, `init_scale in {0.001, 0.01, 0.1, 1, 2}` at size `50` as the baseline comparison

It also shows how to compute **paper-faithful interference** from the generated fit CSVs:

- `interference = 1 - A_weight_A2`

Important note:

- This interference metric is primarily interpretable for `near` and `far`.
- `same` is not expected to support the same mixture-model interpretation.


In [122]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

project_root = Path.cwd()
if project_root.name != "a1b2_modular":
    if (project_root / "a1b2_modular").exists():
        project_root = project_root / "a1b2_modular"
    else:
        for parent in [project_root, *project_root.parents]:
            if parent.name == "a1b2_modular":
                project_root = parent
                break

data_root = project_root / "data" / "simulations"
print(f"project_root = {project_root}")
print(f"data_root    = {data_root}")


project_root = /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular
data_root    = /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations


In [123]:
INIT_LEVELS = [0.001, 0.01, 0.1, 1.0, 2.0]

TARGET_RUNS = [
    {
        "arch_group": "two_module_shared_sp1",
        "routing": "shared",
        "sparsity_label": "1.0",
        "init_scale": 0.001,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_nb2_init0.001_nb2_shared_sp1.0_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "two_module_shared_sp1",
        "routing": "shared",
        "sparsity_label": "1.0",
        "init_scale": 0.01,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_nb2_init0.01_nb2_shared_sp1.0_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "two_module_shared_sp1",
        "routing": "shared",
        "sparsity_label": "1.0",
        "init_scale": 0.1,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_nb2_init0.1_nb2_shared_sp1.0_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "two_module_shared_sp1",
        "routing": "shared",
        "sparsity_label": "1.0",
        "init_scale": 1.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_nb2_nb2_shared_sp1.0_sep_cr_RNN",
    },
    {
        "arch_group": "two_module_shared_sp1",
        "routing": "shared",
        "sparsity_label": "1.0",
        "init_scale": 2.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_nb2_init2_nb2_shared_sp1.0_sep_cr_RNN_init2",
    },
    {
        "arch_group": "two_module_shared_sp05",
        "routing": "shared",
        "sparsity_label": "0.5",
        "init_scale": 0.001,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_sp05_nb2_init0.001_nb2_shared_sp0.5_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "two_module_shared_sp05",
        "routing": "shared",
        "sparsity_label": "0.5",
        "init_scale": 0.01,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_sp05_nb2_init0.01_nb2_shared_sp0.5_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "two_module_shared_sp05",
        "routing": "shared",
        "sparsity_label": "0.5",
        "init_scale": 0.1,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_sp05_nb2_init0.1_nb2_shared_sp0.5_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "two_module_shared_sp05",
        "routing": "shared",
        "sparsity_label": "0.5",
        "init_scale": 1.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_sp05_nb2_nb2_shared_sp0.5_sep_cr_RNN",
    },
    {
        "arch_group": "two_module_shared_sp05",
        "routing": "shared",
        "sparsity_label": "0.5",
        "init_scale": 2.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_sp05_nb2_init2_nb2_shared_sp0.5_sep_cr_RNN_init2",
    },
    {
        "arch_group": "two_module_shared_no_comms",
        "routing": "shared",
        "sparsity_label": "no_comms",
        "init_scale": 0.001,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_no_comms_nb2_init0.001_nb2_shared_sp0_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "two_module_shared_no_comms",
        "routing": "shared",
        "sparsity_label": "no_comms",
        "init_scale": 0.01,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_no_comms_nb2_init0.01_nb2_shared_sp0_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "two_module_shared_no_comms",
        "routing": "shared",
        "sparsity_label": "no_comms",
        "init_scale": 0.1,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_no_comms_nb2_init0.1_nb2_shared_sp0_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "two_module_shared_no_comms",
        "routing": "shared",
        "sparsity_label": "no_comms",
        "init_scale": 1.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_no_comms_nb2_nb2_shared_sp0_sep_cr_RNN",
    },
    {
        "arch_group": "two_module_shared_no_comms",
        "routing": "shared",
        "sparsity_label": "no_comms",
        "init_scale": 2.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_no_comms_nb2_init2_nb2_shared_sp0_sep_cr_RNN_init2",
    },
    {
        "arch_group": "two_module_task_routed_sp1",
        "routing": "task_routed",
        "sparsity_label": "1.0",
        "init_scale": 0.001,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_nb2_init0.001_nb2_task_routed_sp1_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "two_module_task_routed_sp1",
        "routing": "task_routed",
        "sparsity_label": "1.0",
        "init_scale": 0.01,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_nb2_init0.01_nb2_task_routed_sp1_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "two_module_task_routed_sp1",
        "routing": "task_routed",
        "sparsity_label": "1.0",
        "init_scale": 0.1,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_nb2_init0.1_nb2_task_routed_sp1_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "two_module_task_routed_sp1",
        "routing": "task_routed",
        "sparsity_label": "1.0",
        "init_scale": 1.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_nb2_nb2_task_routed_sp1_sep_cr_RNN",
    },
    {
        "arch_group": "two_module_task_routed_sp1",
        "routing": "task_routed",
        "sparsity_label": "1.0",
        "init_scale": 2.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_nb2_init2_nb2_task_routed_sp1.0_sep_cr_RNN_init2",
    },
    {
        "arch_group": "two_module_task_routed_sp05",
        "routing": "task_routed",
        "sparsity_label": "0.5",
        "init_scale": 0.001,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_sp05_nb2_init0.001_nb2_task_routed_sp0.5_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "two_module_task_routed_sp05",
        "routing": "task_routed",
        "sparsity_label": "0.5",
        "init_scale": 0.01,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_sp05_nb2_init0.01_nb2_task_routed_sp0.5_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "two_module_task_routed_sp05",
        "routing": "task_routed",
        "sparsity_label": "0.5",
        "init_scale": 0.1,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_sp05_nb2_init0.1_nb2_task_routed_sp0.5_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "two_module_task_routed_sp05",
        "routing": "task_routed",
        "sparsity_label": "0.5",
        "init_scale": 1.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_sp05_nb2_nb2_task_routed_sp0.5_sep_cr_RNN",
    },
    {
        "arch_group": "two_module_task_routed_sp05",
        "routing": "task_routed",
        "sparsity_label": "0.5",
        "init_scale": 2.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_sp05_nb2_init2_nb2_task_routed_sp0.5_sep_cr_RNN_init2",
    },
    {
        "arch_group": "two_module_task_routed_no_comms",
        "routing": "task_routed",
        "sparsity_label": "no_comms",
        "init_scale": 0.001,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_no_comms_nb2_init0.001_nb2_task_routed_sp0_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "two_module_task_routed_no_comms",
        "routing": "task_routed",
        "sparsity_label": "no_comms",
        "init_scale": 0.01,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_no_comms_nb2_init0.01_nb2_task_routed_sp0_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "two_module_task_routed_no_comms",
        "routing": "task_routed",
        "sparsity_label": "no_comms",
        "init_scale": 0.1,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_no_comms_nb2_init0.1_nb2_task_routed_sp0_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "two_module_task_routed_no_comms",
        "routing": "task_routed",
        "sparsity_label": "no_comms",
        "init_scale": 1.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_no_comms_nb2_nb2_task_routed_sp0_sep_cr_RNN",
    },
    {
        "arch_group": "two_module_task_routed_no_comms",
        "routing": "task_routed",
        "sparsity_label": "no_comms",
        "init_scale": 2.0,
        "comparison_dim_hidden": 25,
        "sim_folder": "two_module_rnn_25_task_routed_no_comms_nb2_init2_nb2_task_routed_sp0_sep_cr_RNN_init2",
    },
    {
        "arch_group": "single_module",
        "routing": "shared",
        "sparsity_label": "single_module",
        "init_scale": 0.001,
        "comparison_dim_hidden": 50,
        "sim_folder": "single_module_rnn_50_nb2_init0.001_nb2_shared_sp1_sep_cr_RNN_init0.001",
    },
    {
        "arch_group": "single_module",
        "routing": "shared",
        "sparsity_label": "single_module",
        "init_scale": 0.01,
        "comparison_dim_hidden": 50,
        "sim_folder": "single_module_rnn_50_nb2_init0.01_nb2_shared_sp1_sep_cr_RNN_init0.01",
    },
    {
        "arch_group": "single_module",
        "routing": "shared",
        "sparsity_label": "single_module",
        "init_scale": 0.1,
        "comparison_dim_hidden": 50,
        "sim_folder": "single_module_rnn_50_nb2_init0.1_nb2_shared_sp1_sep_cr_RNN_init0.1",
    },
    {
        "arch_group": "single_module",
        "routing": "shared",
        "sparsity_label": "single_module",
        "init_scale": 1.0,
        "comparison_dim_hidden": 50,
        "sim_folder": "single_module_rnn_50_nb2_nb2_shared_sp1_sep_cr_RNN",
    },
    {
        "arch_group": "single_module",
        "routing": "shared",
        "sparsity_label": "single_module",
        "init_scale": 2.0,
        "comparison_dim_hidden": 50,
        "sim_folder": "single_module_rnn_50_nb2_init2_nb2_shared_sp1_sep_cr_RNN_init2",
    },
]

targets = pd.DataFrame(TARGET_RUNS)
targets


,arch_group,routing,sparsity_label,init_scale,comparison_dim_hidden,sim_folder
0,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...
1,two_module_shared_sp1,shared,1.0,0.010,25,two_module_rnn_25_nb2_init0.01_nb2_shared_sp1....
2,two_module_shared_sp1,shared,1.0,0.100,25,two_module_rnn_25_nb2_init0.1_nb2_shared_sp1.0...
3,two_module_shared_sp1,shared,1.0,1.000,25,two_module_rnn_25_nb2_nb2_shared_sp1.0_sep_cr_RNN
4,two_module_shared_sp1,shared,1.0,2.000,25,two_module_rnn_25_nb2_init2_nb2_shared_sp1.0_s...
5,two_module_shared_sp05,shared,0.5,0.001,25,two_module_rnn_25_sp05_nb2_init0.001_nb2_share...
6,two_module_shared_sp05,shared,0.5,0.010,25,two_module_rnn_25_sp05_nb2_init0.01_nb2_shared...
7,two_module_shared_sp05,shared,0.5,0.100,25,two_module_rnn_25_sp05_nb2_init0.1_nb2_shared_...
8,two_module_shared_sp05,shared,0.5,1.000,25,two_module_rnn_25_sp05_nb2_nb2_shared_sp0.5_se...
9,two_module_shared_sp05,shared,0.5,2.000,25,two_module_rnn_25_sp05_nb2_init2_nb2_shared_sp...


## 1. Audit whether the fitting completed correctly

A run is considered successfully completed here if:

- the simulation folder exists
- the folder contains participant `.npz` files
- the generated `*_vonmises_fits.csv` exists
- the CSV has the expected participant count and required columns


In [124]:
required_fit_cols = {
    "participant", "condition",
    "A_weight_A1", "A_weight_B", "A_weight_A2",
    "kappa_A1", "kappa_B", "kappa_A2",
}

audit_rows = []
for row in TARGET_RUNS:
    sim_folder = row["sim_folder"]
    sim_dir = data_root / sim_folder
    fit_csv = data_root / f"{sim_folder}_vonmises_fits.csv"
    npz_files = sorted(sim_dir.glob("*.npz")) if sim_dir.exists() else []

    info = dict(row)
    info["sim_dir_exists"] = sim_dir.exists()
    info["n_npz_files"] = len(npz_files)
    info["fit_csv_exists"] = fit_csv.exists()
    info["fit_csv_path"] = str(fit_csv)

    if fit_csv.exists():
        fit_df = pd.read_csv(fit_csv)
        info["n_fit_rows"] = len(fit_df)
        info["n_fit_participants"] = fit_df["participant"].nunique() if "participant" in fit_df else np.nan
        info["fit_conditions"] = sorted(fit_df["condition"].dropna().unique().tolist()) if "condition" in fit_df else []
        info["missing_required_cols"] = sorted(required_fit_cols.difference(fit_df.columns))
        info["fit_complete"] = (
            len(npz_files) > 0
            and len(required_fit_cols.difference(fit_df.columns)) == 0
            and fit_df["participant"].nunique() > 0
        )
    else:
        info["n_fit_rows"] = np.nan
        info["n_fit_participants"] = np.nan
        info["fit_conditions"] = []
        info["missing_required_cols"] = []
        info["fit_complete"] = False

    audit_rows.append(info)

audit_df = pd.DataFrame(audit_rows)
audit_df


,arch_group,routing,sparsity_label,init_scale,comparison_dim_hidden,sim_folder,sim_dir_exists,n_npz_files,fit_csv_exists,fit_csv_path,n_fit_rows,n_fit_participants,fit_conditions,missing_required_cols,fit_complete
0,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
1,two_module_shared_sp1,shared,1.0,0.010,25,two_module_rnn_25_nb2_init0.01_nb2_shared_sp1....,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
2,two_module_shared_sp1,shared,1.0,0.100,25,two_module_rnn_25_nb2_init0.1_nb2_shared_sp1.0...,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
3,two_module_shared_sp1,shared,1.0,1.000,25,two_module_rnn_25_nb2_nb2_shared_sp1.0_sep_cr_RNN,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
4,two_module_shared_sp1,shared,1.0,2.000,25,two_module_rnn_25_nb2_init2_nb2_shared_sp1.0_s...,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
5,two_module_shared_sp05,shared,0.5,0.001,25,two_module_rnn_25_sp05_nb2_init0.001_nb2_share...,True,202,False,/home/kat/workspace/Structure-Function-Analysi...,NaN,NaN,[],[],False
6,two_module_shared_sp05,shared,0.5,0.010,25,two_module_rnn_25_sp05_nb2_init0.01_nb2_shared...,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
7,two_module_shared_sp05,shared,0.5,0.100,25,two_module_rnn_25_sp05_nb2_init0.1_nb2_shared_...,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
8,two_module_shared_sp05,shared,0.5,1.000,25,two_module_rnn_25_sp05_nb2_nb2_shared_sp0.5_se...,True,305,True,/home/kat/workspace/Structure-Function-Analysi...,202.0,202.0,"[far, near]",[],True
9,two_module_shared_sp05,shared,0.5,2.000,25,two_module_rnn_25_sp05_nb2_init2_nb2_shared_sp...,True,248,False,/home/kat/workspace/Structure-Function-Analysi...,NaN,NaN,[],[],False


In [125]:
summary_cols = [
    "arch_group", "routing", "sparsity_label", "init_scale",
    "n_npz_files", "fit_csv_exists", "n_fit_participants",
    "fit_conditions", "missing_required_cols", "fit_complete",
]
audit_df[summary_cols].sort_values(["arch_group", "init_scale"])


,arch_group,routing,sparsity_label,init_scale,n_npz_files,fit_csv_exists,n_fit_participants,fit_conditions,missing_required_cols,fit_complete
30,single_module,shared,single_module,0.001,305,True,202.0,"[far, near]",[],True
31,single_module,shared,single_module,0.010,305,True,202.0,"[far, near]",[],True
32,single_module,shared,single_module,0.100,305,True,202.0,"[far, near]",[],True
33,single_module,shared,single_module,1.000,305,True,202.0,"[far, near]",[],True
34,single_module,shared,single_module,2.000,305,True,202.0,"[far, near]",[],True
10,two_module_shared_no_comms,shared,no_comms,0.001,305,True,202.0,"[far, near]",[],True
11,two_module_shared_no_comms,shared,no_comms,0.010,305,True,202.0,"[far, near]",[],True
12,two_module_shared_no_comms,shared,no_comms,0.100,305,True,202.0,"[far, near]",[],True
13,two_module_shared_no_comms,shared,no_comms,1.000,305,True,202.0,"[far, near]",[],True
14,two_module_shared_no_comms,shared,no_comms,2.000,305,True,202.0,"[far, near]",[],True


## 2. Load completed fit CSVs

Each fit CSV contains participant-level von Mises estimates. The key paper-faithful quantity is the A2 mixture weight:

- `A_weight_A2`: estimated probability mass on Rule A during A2
- `interference = 1 - A_weight_A2`: estimated Rule-B usage during A2


In [126]:
fit_frames = []
for row in TARGET_RUNS:
    fit_csv = data_root / f"{row['sim_folder']}_vonmises_fits.csv"
    if not fit_csv.exists():
        continue
    df = pd.read_csv(fit_csv).copy()
    for key, value in row.items():
        df[key] = value
    fit_frames.append(df)

fits = pd.concat(fit_frames, ignore_index=True) if fit_frames else pd.DataFrame()
fits.shape


(6185, 14)

In [127]:
if fits.empty:
    print("No fit CSVs found yet.")
else:
    display(fits.head())


,participant,condition,A_weight_A1,kappa_A1,A_weight_B,kappa_B,A_weight_A2,kappa_A2,arch_group,routing,sparsity_label,init_scale,comparison_dim_hidden,sim_folder
0,sim_study2_near_sub38,near,0.956597,23.417899,0.10000,927.085537,0.000003,58.557561,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...
1,sim_study1_near_sub50,near,0.763146,7.429183,0.10000,696.735500,0.000020,53.431221,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...
2,sim_study2_near_sub66,near,0.656099,3.460328,0.09948,430.402809,0.000080,41.507753,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...
3,sim_study1_near_sub47,near,0.966179,1.950195,0.10000,503.363211,0.000048,42.370741,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...
4,sim_study2_near_sub30,near,0.996489,4.781103,0.08312,283.693228,0.000007,54.176539,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...


## 3. Compute paper-faithful interference

For each participant and condition:

- `interference = 1 - A_weight_A2`

This notebook keeps `same` visible if present in the fit CSV, but the primary interpretation should focus on `near` and `far`.


In [128]:
if fits.empty:
    interference_df = pd.DataFrame()
else:
    interference_df = fits.copy()
    interference_df["interference"] = 1.0 - interference_df["A_weight_A2"]
    interference_df["interference_clipped"] = interference_df["interference"].clip(0.0, 1.0)

interference_df.head() if not interference_df.empty else interference_df


,participant,condition,A_weight_A1,kappa_A1,A_weight_B,kappa_B,A_weight_A2,kappa_A2,arch_group,routing,sparsity_label,init_scale,comparison_dim_hidden,sim_folder,interference,interference_clipped
0,sim_study2_near_sub38,near,0.956597,23.417899,0.10000,927.085537,0.000003,58.557561,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,0.999997,0.999997
1,sim_study1_near_sub50,near,0.763146,7.429183,0.10000,696.735500,0.000020,53.431221,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,0.999980,0.999980
2,sim_study2_near_sub66,near,0.656099,3.460328,0.09948,430.402809,0.000080,41.507753,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,0.999920,0.999920
3,sim_study1_near_sub47,near,0.966179,1.950195,0.10000,503.363211,0.000048,42.370741,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,0.999952,0.999952
4,sim_study2_near_sub30,near,0.996489,4.781103,0.08312,283.693228,0.000007,54.176539,two_module_shared_sp1,shared,1.0,0.001,25,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,0.999993,0.999993


In [129]:
if interference_df.empty:
    print("No completed fits available.")
else:
    near_far = interference_df[interference_df["condition"].isin(["near", "far"])].copy()
    summary = (
        near_far.groupby(["arch_group", "routing", "sparsity_label", "init_scale", "condition"], observed=True)
        .agg(
            n=("participant", "nunique"),
            A_weight_A2_mean=("A_weight_A2", "mean"),
            A_weight_A2_sem=("A_weight_A2", "sem"),
            interference_mean=("interference", "mean"),
            interference_sem=("interference", "sem"),
            kappa_A2_mean=("kappa_A2", "mean"),
            kappa_A2_sem=("kappa_A2", "sem"),
        )
        .reset_index()
        .sort_values(["arch_group", "init_scale", "condition"])
    )
    display(summary)


,arch_group,routing,sparsity_label,init_scale,condition,n,A_weight_A2_mean,A_weight_A2_sem,interference_mean,interference_sem,kappa_A2_mean,kappa_A2_sem
0,single_module,shared,single_module,0.001,far,101,0.392826,2.700984e-02,6.071742e-01,2.700984e-02,0.943797,0.082341
1,single_module,shared,single_module,0.001,near,101,0.000017,3.867943e-06,9.999835e-01,3.867943e-06,55.268888,0.606493
2,single_module,shared,single_module,0.010,far,101,0.999807,1.663275e-04,1.928849e-04,1.663275e-04,27.093603,6.758280
3,single_module,shared,single_module,0.010,near,101,0.000018,4.274738e-06,9.999822e-01,4.274738e-06,55.014948,0.605364
4,single_module,shared,single_module,0.100,far,101,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,128.905907,11.675895
5,single_module,shared,single_module,0.100,near,101,0.000105,4.719924e-05,9.998948e-01,4.719924e-05,50.610753,0.602771
6,single_module,shared,single_module,1.000,far,101,0.938435,7.783587e-03,6.156523e-02,7.783587e-03,13.905894,1.751525
7,single_module,shared,single_module,1.000,near,101,0.835181,1.890222e-02,1.648186e-01,1.890222e-02,26.870449,1.460477
8,single_module,shared,single_module,2.000,far,101,0.933902,8.638196e-03,6.609793e-02,8.638196e-03,5.008628,0.395068
9,single_module,shared,single_module,2.000,near,101,0.781788,2.637900e-02,2.182122e-01,2.637900e-02,4.610208,0.797289


## 4. Sanity checks

Useful basic checks:

- `A_weight_A2` should usually lie near `[0, 1]`
- `interference = 1 - A_weight_A2` should usually lie near `[0, 1]`
- `near` and `far` should both be present for the ANN runs
- participant counts should be roughly consistent across compared cells


In [130]:
if interference_df.empty:
    print("No completed fits available.")
else:
    checks = (
        interference_df.groupby(["arch_group", "init_scale", "condition"], observed=True)
        .agg(
            n=("participant", "nunique"),
            min_A_weight_A2=("A_weight_A2", "min"),
            max_A_weight_A2=("A_weight_A2", "max"),
            min_interference=("interference", "min"),
            max_interference=("interference", "max"),
        )
        .reset_index()
        .sort_values(["arch_group", "init_scale", "condition"])
    )
    display(checks)


,arch_group,init_scale,condition,n,min_A_weight_A2,max_A_weight_A2,min_interference,max_interference
0,single_module,0.001,far,101,6.407656e-02,0.945947,5.405253e-02,9.359234e-01
1,single_module,0.001,near,101,2.778062e-07,0.000255,9.997452e-01,9.999997e-01
2,single_module,0.010,far,101,9.834206e-01,1.000000,0.000000e+00,1.657940e-02
3,single_module,0.010,near,101,2.957274e-07,0.000252,9.997485e-01,9.999997e-01
4,single_module,0.100,far,101,1.000000e+00,1.000000,0.000000e+00,0.000000e+00
5,single_module,0.100,near,101,3.151398e-07,0.003490,9.965099e-01,9.999997e-01
6,single_module,1.000,far,101,8.333174e-01,1.000000,0.000000e+00,1.666826e-01
7,single_module,1.000,near,101,2.186414e-01,0.999966,3.389203e-05,7.813586e-01
8,single_module,2.000,far,101,6.553369e-01,1.000000,0.000000e+00,3.446631e-01
9,single_module,2.000,near,101,8.227528e-02,0.999345,6.545605e-04,9.177247e-01


## 5. Paper-faithful output table for later notebook merge

This is the participant-level table you can merge back into a figure notebook.

Recommended merge keys:

- `participant`
- `condition`
- plus scenario metadata such as `sim_folder` or (`arch_group`, `init_scale`, `sparsity_label`, `routing`)


In [131]:
if interference_df.empty:
    paper_faithful_interference = pd.DataFrame()
else:
    paper_faithful_interference = interference_df[
        [
            "participant", "condition",
            "arch_group", "routing", "sparsity_label", "init_scale", "sim_folder",
            "A_weight_A2", "kappa_A2", "interference",
        ]
    ].copy()
    paper_faithful_interference = paper_faithful_interference.sort_values(
        ["arch_group", "init_scale", "condition", "participant"]
    )

paper_faithful_interference.head(20)


,participant,condition,arch_group,routing,sparsity_label,init_scale,sim_folder,A_weight_A2,kappa_A2,interference
5342,sim_study1_far_sub1,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.842960,0.044213,0.157040
5322,sim_study1_far_sub10,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.854165,0.032577,0.145835
5329,sim_study1_far_sub12,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.455691,1.245715,0.544309
5316,sim_study1_far_sub13,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.131326,0.132679,0.868674
5302,sim_study1_far_sub15,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.470518,0.741167,0.529482
5287,sim_study1_far_sub16,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.071339,0.661771,0.928661
5324,sim_study1_far_sub19,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.341964,1.848712,0.658036
5349,sim_study1_far_sub20,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.142186,0.064794,0.857814
5325,sim_study1_far_sub21,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.848652,0.171076,0.151348
5355,sim_study1_far_sub22,far,single_module,shared,single_module,0.001,single_module_rnn_50_nb2_init0.001_nb2_shared_...,0.345926,2.072103,0.654074


## 6. Optional export

Uncomment the save line if you want one combined CSV for the tested primary grid.


In [132]:
export_path = project_root / "data" / "simulations" / "size25_taskrouted_primary_vonmises_interference_summary.csv"
# paper_faithful_interference.to_csv(export_path, index=False)
export_path


PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/size25_taskrouted_primary_vonmises_interference_summary.csv')